In [20]:
from __init__ import PRP; import sys
sys.path.append(PRP + 'veros/')

from datetime import datetime
from jax import config
config.update("jax_enable_x64", True)

import jax
sys.path.append(PRP)

from scripts.load_runtime import * #Setup parameters for veros 
from setups.acc.acc_learning import ACCSetup

import jax.numpy as jnp
from jax import vmap

from tqdm import tqdm

In [21]:
from functools import partial

# Spin-Up

In [22]:
# Spin-up 
warmup_steps = 200
acc = ACCSetup()
acc.setup()
with acc.state.settings.unlock() :
    acc.state.settings.enable_eke = False

with acc.state.variables.unlock() :
     acc.state.variables.r_bot += 1e-5
     acc.state.variables.K_gm_0 += 1000.0

def ps(state) : 
    #n_state = state.copy()
    acc.step(state)
    return state

step_jit = jax.jit(ps)

state = acc.state.copy()
for step in tqdm(range(warmup_steps)) :
    state = step_jit(state)

acc.state = state

Running model setup
Diffusion grid factor delta_iso1 = 0.01942284820457075


100%|██████████| 200/200 [00:03<00:00, 55.78it/s] 


# Compute derivative

In [ ]:
class autodiff() :
    def __init__(self, step_function, agg_function,  var_name) :
        """
            Computes derivative dL/dvar with L in R and var in R
            step_function is the function done n iterations
            agg_function is computed at the end to go from R^space -> R
            var_name : name of the variable in the state to differentiatiat w.r.t

            Rollout is done with jax.lax.scan instead of a Python for-loop:
            state -> lax.scan(step, state, length=iterations) -> state
            This traces the step function once and reuses it, instead of
            unrolling it `iterations` times.
        """
        self.agg_function = agg_function
        self.step_function = partial(autodiff.pure, step=step_function)
        self.var_name = var_name
        self._grad_fn_cache = {}

    @staticmethod
    def pure(state, step) :
        """
            Convert the state function into a "pure step" copying the input state
        """
        n_state = state.copy()
        step(n_state)  # This is a function that modifies state object inplace
        return n_state

    @staticmethod
    def set_var(var_name, state, var_value):
        n_state = state.copy()
        vs = n_state.variables
        with n_state.variables.unlock():
            setattr(vs, var_name, var_value)
        return n_state

    def rollout(self, n_state, iterations) :
        # scan carries the state forward `iterations` times; no per-step
        # output is collected (carry, None), only the final state is used
        n_state, _ = jax.lax.scan(
            lambda c, _: (self.step_function(c), None),
            n_state,
            length=iterations,
        )
        return n_state

    def g(self, state, var_value, iterations=1, **kwargs):
        # `iterations` is static (baked into the scan length), so cache
        # one compiled grad function per value -- see class docstring
        if iterations not in self._grad_fn_cache:
            def loss_fn(v, s):
                n_state = autodiff.set_var(self.var_name, s, v)
                n_state = self.rollout(n_state, iterations)
                return self.agg_function(n_state)
            self._grad_fn_cache[iterations] = jax.jit(jax.value_and_grad(loss_fn))

        loss, grad = self._grad_fn_cache[iterations](var_value, state)
        return loss, grad

In [ ]:
def agg_function(state) :
    return (state.variables.temp ** 2).sum()

var_dev = 'K_gm_0'
it = 2

In [ ]:
vjpm_nr = autodiff(acc.step, agg_function, var_dev)

vjpm_nr.step_function = jax.jit(vjpm_nr.step_function)
vjpm_nr.agg_function = jax.jit(vjpm_nr.agg_function)

vjpm_nr.step_function = jax.checkpoint(vjpm_nr.step_function) # Remat to save memory


loss_and_grad_nr = lambda s, v, it: vjpm_nr.g(s, v, iterations=it)

In [ ]:
loss_and_grad_nr(acc.state, 1e-5, 2)

# Parameter fitting experiment 

We try to fit a variable ("r_bot" or other) so that the state after it is similar to a target state. 

In [ ]:
pred_iter = 5
targe_state = vjpm_nr.rollout(acc.state, iterations=pred_iter)
# We fix the target state iterating from the initial configuration

In [ ]:
def agg_function(state) :
    return ((state.variables.temp - targe_state.variables.temp) ** 2).sum()

In [ ]:
vjpm_nr = autodiff(acc.step, agg_function, var_dev)

vjpm_nr.step_function = jax.jit(vjpm_nr.step_function)
vjpm_nr.agg_function = jax.jit(vjpm_nr.agg_function)

vjpm_nr.step_function = jax.checkpoint(vjpm_nr.step_function) # Remat to save memory


loss_and_grad_nr = lambda s, v, it: vjpm_nr.g(s, v, iterations=it)

In [ ]:
loss_and_grad_nr(acc.state, 0.0, pred_iter)

In [ ]:
params = jnp.linspace(800, 1200, 10) #jnp.linspace(-4e-5, 4e-5, 10) (for rbots)

In [ ]:
losses = []
grads = []
for pr in params :
    loss, grad = loss_and_grad_nr(acc.state, pr, pred_iter)
    losses.append(loss)
    grads.append(grad)

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(15,5))
fig.suptitle(f'{var_dev}')
axs[0].set_title('Loss')
axs[0].plot(params, losses)
axs[0].axvline(getattr(acc.state.variables, var_dev), color='r', linestyle='dashed')
axs[1].set_title('Gradient')
axs[1].plot(params, grads)
axs[1].axvline(getattr(acc.state.variables, var_dev), color='r', linestyle='dashed')

# Visualize gradients

In [ ]:
vjpm_nr = autodiff(acc.step, agg_function, 'temp')


vjpm_nr.step_function = jax.jit(vjpm_nr.step_function)
vjpm_nr.agg_function = jax.jit(vjpm_nr.agg_function)

vjpm_nr.step_function = jax.checkpoint(vjpm_nr.step_function) # Remat to save memory


loss_and_grad_nr = lambda s, v, it: vjpm_nr.g(s, v, iterations=it)

In [ ]:
rollout_lengths = [1, 2, 5, 10]

losses_by_iter, grads_by_iter = {}, {}
for it in rollout_lengths:
    l, g = loss_and_grad_nr(acc.state, acc.state.variables.temp, it)
    losses_by_iter[it] = l
    grads_by_iter[it] = g

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

vs = acc.state.variables

# xt/yt carry a 2-cell ghost/halo ring on each side (used for cyclic BCs) -- trim it
# so we only plot the physical domain
sl = slice(2, -2)
lon = np.asarray(vs.xt[sl])
lat = np.asarray(vs.yt[sl])
depth = np.asarray(vs.zt)                      # zt[0] = deepest layer, zt[-1] = surface
land_mask = np.asarray(vs.maskT[sl, sl, :], dtype=bool)

z_levels = {"bottom": 0, "mid-depth": len(depth) // 2, "surface": -1}

# g has shape (xt, yt, zt, tau); use vs.tau (the "current" time slot, same convention
# as e.g. vs.temp[:, :, -1, vs.tau] elsewhere) instead of hardcoding slot 0.
grads_trimmed = {
    it: np.where(land_mask, np.asarray(g[sl, sl, :, vs.tau]), np.nan)
    for it, g in grads_by_iter.items()
}


def symmetric_norm(data):
    vmax = np.nanmax(np.abs(data))
    vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1e-12
    return TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)


fig, axs = plt.subplots(
    len(z_levels), len(rollout_lengths),
    figsize=(4.5 * len(rollout_lengths), 3.5 * len(z_levels)),
    sharex=True, sharey=True, constrained_layout=True,
)
for row, (z_label, k) in zip(axs, z_levels.items()):
    for ax, it in zip(row, rollout_lengths):
        data = grads_trimmed[it][:, :, k]
        # each panel gets its own dynamic range -- with a shared scale, the small
        # early-rollout gradients get washed out by the much larger later ones
        im = ax.pcolormesh(lon, lat, data.T, cmap="RdBu_r", norm=symmetric_norm(data), shading="auto")
        ax.set_facecolor("0.8")  # land (nan) shows through as gray
        if z_label == next(iter(z_levels)):
            ax.set_title(f"{it} step{'s' if it > 1 else ''}")
        fig.colorbar(im, ax=ax, shrink=0.85)
    row[0].set_ylabel(f"{z_label}\nlatitude (deg)")
for ax in axs[-1]:
    ax.set_xlabel("longitude (deg)")

fig.suptitle("Evolution of the spatial gradient d(loss)/d(temp) with rollout length")
plt.show()

In [ ]:
# Zonal-mean (averaged over longitude) latitude-depth section, to see the vertical
# structure of the gradient that the horizontal maps above don't show -- again one
# panel per rollout length, each with its own dynamic range and colorbar
zonal_mean_by_iter = {it: np.nanmean(gr, axis=0) for it, gr in grads_trimmed.items()}  # each (yt, zt)

fig, axs = plt.subplots(1, len(rollout_lengths), figsize=(4.5 * len(rollout_lengths), 4), sharey=True, constrained_layout=True)
for ax, it in zip(axs, rollout_lengths):
    data = zonal_mean_by_iter[it]
    im = ax.pcolormesh(lat, depth, data.T, cmap="RdBu_r", norm=symmetric_norm(data), shading="auto")
    ax.set_facecolor("0.8")
    ax.set_title(f"{it} step{'s' if it > 1 else ''}")
    ax.set_xlabel("latitude (deg)")
    fig.colorbar(im, ax=ax, shrink=0.85)
axs[0].set_ylabel("depth (m)")

fig.suptitle("Evolution of the zonal-mean spatial gradient d(loss)/d(temp) with rollout length")
plt.show()